In [ ]:
import sys
sys.path.insert(0, '../code')
import paths  # input and output locations, from config.yaml
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.plot import show
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.pyplot as plt

In [ ]:
import matplotlib
matplotlib.use("Agg")
from palettable.scientific.sequential import Batlow_9_r
cmap = Batlow_9_r.mpl_colormap
threshold = 10
# Switched from 50km to 25km blocks for consistency with Fig.1/1w
# (see Supplementary Fig. S2 for the resolution sensitivity comparison).
output_path = paths.der(f'rasters/pop_{threshold}_block25x25.tiff')
src = rasterio.open(output_path)
data_pt = src.read(1)
data_pt = np.where(data_pt == -9999, np.nan, data_pt)

# Define color boundaries: dense between 0-30, sparse above
#data_pt = data_pt/1e3

#bounds = list(np.arange(0,101, 10)) + list(np.arange(1000,100001, 1000)) + list(np.arange(100000,1000001, 100000)) + \
#list(np.arange(1000000,2500001, 100000))

bounds = [0, 100, 1000, 10000, 100000, 250000, 500000, 1000000]
norm = mcolors.BoundaryNorm(bounds, ncolors=256)

#norm = BoundaryNorm(boundaries=bounds, ncolors=cmap.N, extend='both')
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])



fig = plt.figure(figsize=(10, 10))
gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[2, 1.2])
# Map subplot
ax1 = fig.add_subplot(gs[0], projection=ccrs.PlateCarree())
ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
ax1.add_feature(cfeature.OCEAN, facecolor='white')
ax1.add_feature(cfeature.LAND, facecolor='#e1dfdf')
show(data_pt , transform=src.transform, ax=ax1, cmap=cmap, norm=norm)
gl = ax1.gridlines(draw_labels=True)
gl.xlines = gl.ylines = False
gl.top_labels = gl.right_labels = False
ax1.text(.01, .975, 'A', transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top')

# Create a new axes for the horizontal colorbar inside ax1
cbar_ax = fig.add_axes([0.31, 0.5, 0.4, 0.01])  
# Create horizontal colorbar in that axes
# Crea la colorbar con i bounds
cbar = plt.colorbar(sm, cax=cbar_ax, orientation='horizontal', boundaries=bounds)
cbar.set_ticklabels(['0', '100', '1K', '10K', '100k', '250K', '500K', '1M'])

cbar.set_label('Population living in 10 minutes to forest [min]', fontsize=12, labelpad=5)
cbar.ax.tick_params(labelsize=10, direction='out')
cbar.ax.xaxis.set_label_position('top')  # Title above
cbar.ax.xaxis.tick_bottom()



import seaborn as sns
from palettable.cartocolors.diverging import Geyser_5_r, Geyser_5
from palettable.cartocolors.qualitative import Safe_3, Antique_5, Vivid_7
from palettable.cubehelix import cubehelix3_16
dfTR = pd.read_csv(paths.der("fig2b.csv"))
import sys
sys.path.insert(0, '../code')
from constants import CONTINENT_COLORS, REGION_TO_CONTINENT
from palettable.scientific.sequential import Davos_5
fname = paths.der("population_by_macroregion_traveltime.csv")
df = pd.read_csv(fname)
df_sum = df.groupby('threshold_min')['population'].sum().reset_index()
df_sum['pop_rel'] = 100*df_sum['population']/7830089931
df_sum['population'] = df_sum['population'] / 1e9
# Barplot subplot




ax2 = fig.add_subplot(gs[1])

sns.barplot(x='threshold_min', y='rel_pop', hue='continent', ax=ax2, data=dfTR, palette=CONTINENT_COLORS, alpha=0.75)

#bars = ax2.bar(range(len(df_sum['threshold_min'])), df_sum['population'], color=Davos_5.hex_colors[2])

ax2.set_ylabel('Population [%]', size=12)
ax2.set_xlabel('Travel time [min]', size=12)
ax2.set_xticks(range(len(df_sum['threshold_min'])))
ax2.set_xticklabels(df_sum['threshold_min'], rotation=0, ha='right')
ax2.text(0.01, .975, 'B', transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top')

# Linea cumulativa

ax2.plot(range(len(df_sum['threshold_min'])), df_sum['pop_rel'], color='black', lw=3, ls='--', label='Global population')

# Trova l'indice più vicino al 50%
threshold_index_50 = (df_sum['pop_rel'] - 50).abs().idxmin()
x_50 = threshold_index_50
y_50 = df_sum['pop_rel'].iloc[x_50]
x_val = df_sum['threshold_min'].iloc[x_50]

# Aggiungi una linea verticale e annotazione
ax2.axhline(50, color='crimson', linestyle='--', lw=1)
ax2.axvline(x_50, color='crimson', linestyle='--', lw=1)
ax2.plot(x_50, y_50, 'o', color='crimson')
#ax2.annotate(f'50% at {x_val} min',
#             xy=(x_50, y_50), xycoords='data',
#             xytext=(-70, 80), textcoords='offset points',
#             arrowprops=dict(arrowstyle='->', color='crimson'),
#             fontsize=12, color='crimson')

ax2.legend(title=' ',
           title_fontsize=1,
           fontsize=10,
            bbox_to_anchor=(1.01, 1), 
           loc='upper left',
           frameon=True)

In [ ]:
fout = os.path.join(paths.fig("Figure2AB.pdf"))
fig.savefig(fout, dpi=300, bbox_inches='tight')

In [ ]:
block_size = (50, 50)
block_size[0]